## Load deps

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

In [5]:
load_dotenv()

True

## Create OpenAI client

Check that the OpenAI client works

In [6]:
openai_client = OpenAI()

## Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

This is our black box - text goes in, text comes out.

Let's test it:

In [8]:
llm("Hey, what's up?")

'Not much — just here and ready to help. What’s on your mind?'

Ask it a course-specific question:

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes—usually you can join if the course is still open for enrollment.

If you want, send me:
- the course name
- where you found it
- the start date (if you know it)

and I can help you figure out whether it’s likely still possible to join and what to do next.


This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. But our courses are not in the training data.

## Adding context manually

More context can fix this. The FAQ website has questions and answers about our courses.

Copy some of that content into the prompt:

In [10]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


## Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```
That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:
-   search
-   prompt
-   LLM


The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.

The database and the LLM can be anything. Because each piece is independent, RAG stays flexible.
- for now, we use minsearch and then sqlitesearch for search, and OpenAI for the LLM
- To use Anthropic instead of OpenAI, you swap the LLM call.
- To use Elasticsearch instead of minsearch, you swap the search call. Nothing else changes.